# 59R2 complete annotated_only shadow test

Run all in Colab. Tests the frozen 59R2 bundle against the complete 60R2 reviewed cohort and writes 59R1-versus-59R2 page outputs. No Word documents are created or copied.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib joblib scikit-learn


In [ ]:
from pathlib import Path
from datetime import datetime
import json, hashlib, gc, zipfile
import cv2, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
R60_OVERRIDE=None; R59R2_OVERRIDE=None; EXPECTED_PAGES=151; BATCH=250000; FOCUS=['Page258']
RUN=ROOT/'59r2_annotated_only_shadow_test'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_59r2_shadow')
T=RUN/'tables'; P=RUN/'pages'; A=RUN/'audit'; V=RUN/'charts'
for d in [RUN,T,P,A,V]: d.mkdir(parents=True,exist_ok=False)
def req(x,m):
    if not x: raise RuntimeError(m)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def savej(p,x): p.write_text(json.dumps(x,indent=2,default=str))
def runs(required):
    return sorted([p for p in ROOT.rglob('*') if p.is_dir() and all((p/r).is_file() for r in required)],key=lambda p:p.stat().st_mtime)
R60=Path(R60_OVERRIDE) if R60_OVERRIDE else runs(['tables/pixel_predictions.parquet'])[-1]
required=['bundle/policy.json','bundle/predictor_schema.json','bundle/remove_model.joblib','bundle/remove_calibrator.joblib','bundle/runtime_contract.json','bundle/freeze_manifest.json']
R2=Path(R59R2_OVERRIDE) if R59R2_OVERRIDE else runs(required)[-1]; B=R2/'bundle'
policy=json.loads((B/'policy.json').read_text()); schema=json.loads((B/'predictor_schema.json').read_text()); contract=json.loads((B/'runtime_contract.json').read_text())
checks={'model':policy.get('model_sha256')==sha(B/'remove_model.joblib'),'calibrator':policy.get('calibrator_sha256')==sha(B/'remove_calibrator.joblib'),'schema':policy.get('schema_sha256')==sha(B/'predictor_schema.json'),'policy':policy.get('policy_id')==contract.get('policy_id'),'compatibility':policy.get('compatibility_id')==contract.get('compatibility_id'),'identity_not_predictor':policy.get('component_identity_is_predictor') is False}
req(all(checks.values()),f'Bundle failure: {checks}'); savej(A/'bundle_checks.json',checks)
model=joblib.load(B/'remove_model.joblib'); cal=joblib.load(B/'remove_calibrator.joblib'); pred=list(schema['ordered_predictors'])
th=float(contract.get('threshold',policy.get('threshold'))); conn=int(contract['connectivity']); minrun=int(contract['minimum_run']); region=float(contract['region_mean_threshold'])
df=pd.read_parquet(R60/'tables/pixel_predictions.parquet'); df['page']=df.page.astype(str); df['x']=df.x.astype(np.int32); df['y']=df.y.astype(np.int32); df['component_key']=df.component_id.astype(np.int32)
pages=sorted(df.page.unique()); req(EXPECTED_PAGES is None or len(pages)==EXPECTED_PAGES,f'Expected {EXPECTED_PAGES} pages, found {len(pages)}')
PIX=['observed_value','normalised_value','paper_value','mask_depth','boundary_distance','distance_to_protected','edge','local_edge_fraction','local_dark_fraction','local_preserve_fraction','local_remove_fraction','local_ambiguous_fraction','local_horizontal_fraction','local_vertical_fraction','local_compact_fraction','p_print']
COMP=['component_area','component_width','component_height','component_aspect_ratio','component_fill_ratio','component_touches_page_border','component_maximum_mask_depth','component_mean_mask_depth','component_median_mask_depth','component_preserve_fraction','component_remove_fraction','component_ambiguous_fraction','component_horizontal_fraction','component_vertical_fraction','component_compact_fraction','component_mean_edge_fraction','component_mean_dark_fraction','component_mean_distance_to_protected']
req(set(pred)==set(PIX)|set(COMP),'Frozen schema differs from 34-feature contract')
for c in PIX: df[c]=pd.to_numeric(df[c],errors='coerce').astype(np.float32)
sem=pd.MultiIndex.from_arrays([df.page,df.component_key]); codes,_=pd.factorize(sem,sort=True); codes=codes.astype(np.int32)
w=pd.DataFrame({'code':codes,'page':df.page,'component_key':df.component_key,'x':df.x,'y':df.y,'md':df.mask_depth,'pr':df.local_preserve_fraction,'rr':df.local_remove_fraction,'ar':df.local_ambiguous_fraction,'hr':df.local_horizontal_fraction,'vr':df.local_vertical_fraction,'cr':df.local_compact_fraction,'er':df.local_edge_fraction,'dr':df.local_dark_fraction,'dp':df.distance_to_protected})
g=w.groupby('code',sort=True); cf=g.agg(page=('page','first'),component_key=('component_key','first'),component_area=('x','size'),xmin=('x','min'),xmax=('x','max'),ymin=('y','min'),ymax=('y','max'),component_maximum_mask_depth=('md','max'),component_mean_mask_depth=('md','mean'),component_median_mask_depth=('md','median'),component_preserve_fraction=('pr','mean'),component_remove_fraction=('rr','mean'),component_ambiguous_fraction=('ar','mean'),component_horizontal_fraction=('hr','mean'),component_vertical_fraction=('vr','mean'),component_compact_fraction=('cr','mean'),component_mean_edge_fraction=('er','mean'),component_mean_dark_fraction=('dr','mean'),component_mean_distance_to_protected=('dp','mean')).reset_index()
cf['component_width']=cf.xmax-cf.xmin+1; cf['component_height']=cf.ymax-cf.ymin+1; W=cf.component_width.to_numpy(float); H=cf.component_height.to_numpy(float); AA=cf.component_area.to_numpy(float); cf['component_aspect_ratio']=np.maximum(W,H)/np.maximum(np.minimum(W,H),1); cf['component_fill_ratio']=AA/np.maximum(W*H,1)
SUP={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(d): return {p.stem:p for p in d.iterdir() if p.is_file() and p.suffix.lower() in SUP} if d.exists() else {}
maps=[imap(ROOT/'aligned'),imap(ROOT/'source'/'annotated_only')]
def image_path(page):
    for p in [R60/'pages'/page/'observed.png',R60/'whole_pages'/page/'observed.png']:
        if p.is_file(): return p
    for m in maps:
        if page in m:return m[page]
paths={p:image_path(p) for p in pages}; req(all(paths.values()),'Missing observed page images')
dims=[]
for p,path in paths.items():
    im=cv2.imread(str(path),0); dims.append({'page':p,'height':im.shape[0],'width':im.shape[1],'path':str(path)})
dims=pd.DataFrame(dims); dims.to_csv(A/'resolved_page_images.csv',index=False); cf=cf.merge(dims[['page','height','width']],on='page')
cf['component_touches_page_border']=((cf.xmin<=0)|(cf.ymin<=0)|(cf.xmax>=cf.width-1)|(cf.ymax>=cf.height-1)).astype(np.float32)
for c in COMP: cf[c]=pd.to_numeric(cf[c],errors='raise').astype(np.float32)
cf.to_parquet(T/'component_features.parquet',index=False); pa={c:df[c].to_numpy(np.float32) for c in PIX}; ca={c:cf[c].to_numpy(np.float32) for c in COMP}
raw=np.memmap('/content/r2raw.dat','w+',np.float32,shape=(len(df),)); prob=np.memmap('/content/r2prob.dat','w+',np.float32,shape=(len(df),))
for s in range(0,len(df),BATCH):
    e=min(s+BATCH,len(df)); X=np.empty((e-s,len(pred)),np.float32); cc=codes[s:e]
    for j,c in enumerate(pred): X[:,j]=pa[c][s:e] if c in pa else ca[c][cc]
    r=model.predict_proba(X)[:,1]; p=cal.predict(r); req(np.isfinite(p).all(),'Non-finite probabilities'); raw[s:e]=r; prob[s:e]=p; print(e,'/',len(df)); del X; gc.collect()
raw.flush(); prob.flush(); scored=df[['page','y','x','component_key','paper_value','p_remove','safe_remove']].copy(); scored['raw_remove_59r2']=raw; scored['p_remove_59r2']=prob; scored.to_parquet(T/'59r2_scored_reviewed_cohort.parquet',index=False)
def coherent(y,x,p,h,w):
    q=p>=th; m=np.zeros((h,w),np.uint8); m[y[q],x[q]]=1; n,lab,stats,_=cv2.connectedComponentsWithStats(m,conn); pm=np.full((h,w),np.nan,np.float32); pm[y,x]=p; out=np.zeros((h,w),bool)
    for k in range(1,n):
        z=lab==k; v=pm[z]; v=v[np.isfinite(v)]; size=int(stats[k,cv2.CC_STAT_AREA])
        if size>=minrun and len(v)==size and v.mean()>=region and v.min()>=th: out[z]=1
    return out[y,x]
rows=[]; Y=df.y.to_numpy(); X=df.x.to_numpy(); S=df.safe_remove.astype(bool).to_numpy(); PAPER=df.paper_value.to_numpy(); P2=np.asarray(prob)
for page,ix in df.groupby('page',sort=True).indices.items():
    im=cv2.imread(str(paths[page]),0); y=Y[ix]; x=X[ix]; s1=S[ix]; s2=coherent(y,x,P2[ix],*im.shape); paper=PAPER[ix]; o1=im.copy(); o2=im.copy(); o1[y[s1],x[s1]]=np.clip(np.rint(paper[s1]),0,255).astype(np.uint8); o2[y[s2],x[s2]]=np.clip(np.rint(paper[s2]),0,255).astype(np.uint8); d=P/page; d.mkdir(); cv2.imwrite(str(d/'observed.png'),im); cv2.imwrite(str(d/'59r1_shadow.png'),o1); cv2.imwrite(str(d/'59r2_shadow.png'),o2); m=np.zeros_like(im); m[y[s1!=s2],x[s1!=s2]]=255; cv2.imwrite(str(d/'decision_difference.png'),m); rows.append({'page':page,'pixels':len(ix),'59r1_safe_remove':int(s1.sum()),'59r2_safe_remove':int(s2.sum()),'delta':int(s2.sum()-s1.sum()),'changed_decisions':int((s1!=s2).sum()),'became_safe_remove':int((~s1&s2).sum()),'ceased_safe_remove':int((s1&~s2).sum())}); print(page,rows[-1]['delta'])
res=pd.DataFrame(rows).sort_values('delta'); res.to_csv(T/'59r1_vs_59r2_reviewed_pages.csv',index=False); summary={'pages':len(res),'pixels':int(res.pixels.sum()),'59r1_safe_remove':int(res['59r1_safe_remove'].sum()),'59r2_safe_remove':int(res['59r2_safe_remove'].sum()),'delta':int(res.delta.sum()),'changed_decisions':int(res.changed_decisions.sum()),'promotion_authority':False,'clean_truth_used':False}; savej(RUN/'shadow_test_summary.json',summary)
plt.figure(figsize=(11,max(6,.055*len(res)))); plt.barh(res.page,res.delta); plt.axvline(0,color='black'); plt.xlabel('59R2 minus 59R1 SAFE_REMOVE pixels'); plt.tight_layout(); plt.savefig(V/'delta_by_page.png',dpi=160); plt.close(); savej(RUN/'experiment_identity.json',{'r60':str(R60),'r59r2':str(R2),'policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],'cohort_sha256':sha(R60/'tables/pixel_predictions.parquet'),'pages':len(pages),'rows':len(df),'components':len(cf),'promotion_authority':False}); (RUN/'executive_summary.txt').write_text(json.dumps(summary,indent=2)+'\nSHADOW TEST ONLY. No Word output.\n'); print(json.dumps(summary,indent=2)); print('COMPLETE',RUN)
